# Presets — discovery, dry-run, apply, and rollback

End-to-end walkthrough of the **presets API**:

1. `GET /configs/presets` — list and search available presets.
2. `GET /configs/presets/{name}` — fetch one preset definition.
3. `POST /configs/catalogs/{cat}/presets/{name}/dry-run` — plan without writing.
4. `POST /configs/catalogs/{cat}/presets/{name}` — apply.
5. `GET /configs/presets/applied?scope_key=catalog:{cat}` — read back the audit trail.
6. `DELETE /configs/catalogs/{cat}/presets/{name}` — rollback.

Required env vars:
- `DYNASTORE_BASE_URL` — base URL (default `http://localhost:8080`)
- `DYNASTORE_TOKEN` — bearer token with sysadmin / platform-admin rights

In [ ]:
import json
import os
import time
import uuid

import httpx

try:
    from dotenv import load_dotenv, find_dotenv
    load_dotenv(find_dotenv(usecwd=True), override=False)
except Exception:
    pass

BASE_URL = os.environ.get("DYNASTORE_BASE_URL", "http://localhost:8080")
TOKEN = os.environ.get("DYNASTORE_TOKEN")
if not TOKEN:
    raise RuntimeError("Set DYNASTORE_TOKEN before running.")

RUN_ID = uuid.uuid4().hex[:6]
CAT_ID = f"presets-demo-{RUN_ID}"
PRESET_NAME = "public_open_data"  # catalog-scopable composite preset

JSON_HEADERS = {"Content-Type": "application/json", "Accept": "application/json"}
AUTH_HEADERS = {**JSON_HEADERS, "Authorization": f"Bearer {TOKEN}"}
client = httpx.Client(base_url=BASE_URL, headers=AUTH_HEADERS, timeout=30.0)

print(f"RUN_ID={RUN_ID}  catalog={CAT_ID}  preset={PRESET_NAME}")
print(f"base: {BASE_URL}")

## 1. List available presets

`GET /configs/presets` — returns all registered presets. Supports `tier`, `q`, `name`, `keywords`, `limit`, and cursor pagination.

In [ ]:
r = client.get("/configs/presets", params={"limit": 20})
assert r.is_success, f"list presets: {r.status_code} {r.text[:300]}"
body = r.json()
presets = body.get("presets", [])
print(f"GET /configs/presets: {r.status_code} — {len(presets)} preset(s) returned")
for p in presets:
    print(f"  {p.get('name'):<35}  tier={p.get('tier')}  catalog_scopable={p.get('catalog_scopable')}")

### Filter by tier

In [ ]:
r = client.get("/configs/presets", params={"tier": "platform", "limit": 10})
assert r.is_success, f"filter by tier: {r.status_code} {r.text[:300]}"
platform_presets = r.json().get("presets", [])
print(f"platform-tier presets: {[p.get('name') for p in platform_presets]}")

## 2. Fetch a single preset definition

`GET /configs/presets/{name}` — description, tier, catalog_scopable flag, keywords.

In [ ]:
r = client.get(f"/configs/presets/{PRESET_NAME}")
assert r.is_success, f"get preset: {r.status_code} {r.text[:300]}"
defn = r.json()
print(f"GET /configs/presets/{PRESET_NAME}: {r.status_code}")
print(f"  name             : {defn.get('name')}")
print(f"  tier             : {defn.get('tier')}")
print(f"  catalog_scopable : {defn.get('catalog_scopable')}")
print(f"  keywords         : {defn.get('keywords')}")
print(f"  description      : {str(defn.get('description', ''))[:120]}")

## 3. Create a catalog for the demo

In [ ]:
r = client.post("/stac/catalogs", content=json.dumps({
    "id": CAT_ID, "type": "Catalog", "stac_version": "1.0.0",
    "title": f"Presets demo {RUN_ID}",
    "description": "Presets API demo catalog — safe to delete",
    "links": [],
}))
assert r.status_code in (200, 201, 409), f"catalog create: {r.status_code} {r.text[:200]}"
print(f"POST /stac/catalogs: {r.status_code}")

# Wait for tenant provisioning
deadline = time.monotonic() + 60
while time.monotonic() < deadline:
    r = client.get(f"/stac/catalogs/{CAT_ID}")
    if r.status_code in (200, 401, 403):
        break
    time.sleep(2)
else:
    raise TimeoutError(f"catalog {CAT_ID!r} not reachable after 60s")
print(f"Catalog {CAT_ID!r} ready (HTTP {r.status_code})")

## 4. Dry-run the preset

`POST /configs/catalogs/{cat}/presets/{name}/dry-run` — returns the plan of config slots that would be written, with no side-effects.

In [ ]:
r = client.post(f"/configs/catalogs/{CAT_ID}/presets/{PRESET_NAME}/dry-run")
assert r.is_success, f"dry-run failed: {r.status_code} {r.text[:300]}"
plan = r.json()
print(f"POST /configs/catalogs/{CAT_ID}/presets/{PRESET_NAME}/dry-run: {r.status_code}")
print(f"  preset_name : {plan.get('preset_name')}")
print(f"  scope_key   : {plan.get('scope_key')}")
entries = plan.get("entries", [])
print(f"  entries ({len(entries)}):")
for e in entries:
    print(f"    [{e.get('kind'):<6}] {e.get('target')}")
warnings = plan.get("warnings", [])
if warnings:
    print(f"  warnings: {warnings}")

## 5. Apply the preset

`POST /configs/catalogs/{cat}/presets/{name}` — writes all config slots in the preset's bundle.

In [ ]:
r = client.post(f"/configs/catalogs/{CAT_ID}/presets/{PRESET_NAME}")
assert r.is_success, f"apply preset failed: {r.status_code} {r.text[:300]}"
result = r.json()
print(f"POST /configs/catalogs/{CAT_ID}/presets/{PRESET_NAME}: {r.status_code}")
applied = result.get("applied", [])
skipped = result.get("skipped", [])
print(f"  applied : {len(applied)} slot(s)")
print(f"  skipped : {len(skipped)} slot(s)")
for slot in applied:
    print(f"    + {slot}")

## 6. Read back the applied-presets audit trail

`GET /configs/presets/applied?scope_key=catalog:{cat}` — lists audit rows from `iam.applied_presets`. Requires the IAM extension.

In [ ]:
scope_key = f"catalog:{CAT_ID}"
r = client.get("/configs/presets/applied", params={"scope_key": scope_key})
if r.status_code == 503:
    print(f"GET /configs/presets/applied: 503 — IAM extension not loaded; audit trail unavailable")
else:
    assert r.is_success, f"list applied: {r.status_code} {r.text[:300]}"
    page = r.json()
    rows = page.get("rows", [])
    print(f"GET /configs/presets/applied?scope_key={scope_key}: {r.status_code} — {len(rows)} row(s)")
    for row in rows:
        print(f"  preset={row.get('preset_name')}  state={row.get('state')}  applied_at={row.get('applied_at')}")

## 7. Rollback via DELETE

`DELETE /configs/catalogs/{cat}/presets/{name}` — removes config slots that still match what the preset wrote; diverged slots are left in place.

In [ ]:
r = client.delete(f"/configs/catalogs/{CAT_ID}/presets/{PRESET_NAME}")
assert r.is_success, f"rollback failed: {r.status_code} {r.text[:300]}"
result = r.json()
print(f"DELETE /configs/catalogs/{CAT_ID}/presets/{PRESET_NAME}: {r.status_code}")
deleted = result.get("deleted", [])
skipped = result.get("skipped", [])
print(f"  deleted : {len(deleted)} slot(s)")
print(f"  skipped : {len(skipped)} slot(s)")
for slot in skipped:
    print(f"    ~ {slot.get('slot')}  reason={slot.get('reason')}")

---
## Summary

| Step | Endpoint | Notes |
|---|---|---|
| List | `GET /configs/presets` | filter by `tier`, `q`, `keywords` |
| Fetch | `GET /configs/presets/{name}` | tier, scopability, keywords |
| Dry-run | `POST /configs/catalogs/{cat}/presets/{name}/dry-run` | no side-effects |
| Apply | `POST /configs/catalogs/{cat}/presets/{name}` | writes bundle slots |
| Audit | `GET /configs/presets/applied?scope_key=catalog:{cat}` | IAM extension required |
| Rollback | `DELETE /configs/catalogs/{cat}/presets/{name}` | lenient — diverged slots kept |

## Teardown

In [ ]:
r = client.delete(f"/stac/catalogs/{CAT_ID}", params={"force": "true"})
print(f"DELETE {CAT_ID!r}: {r.status_code}")
client.close()
print("Done.")